In [1]:
import pyomo.environ as pyo

# Item catalog: each entry has a value (utility we get from packing it)
# and a weight (capacity it consumes in the knapsack).
data = {
    'laptop': {
        'value': 25,
        'weight': 6
    },
    'water_bottle': {
        'value': 4,
        'weight': 2
    },
    'tent': {
        'value': 18,
        'weight': 9
    },
    'sleeping_bag': {
        'value': 14,
        'weight': 5
    },
    'flashlight': {
        'value': 6,
        'weight': 1
    },
    'first_aid_kit': {
        'value': 10,
        'weight': 3
    },
    'stove': {
        'value': 12,
        'weight': 4
    },
    'jacket': {
        'value': 11,
        'weight': 4
    },
    'map': {
        'value': 5,
        'weight': 1
    },
    'camera': {
        'value': 16,
        'weight': 3
    }
}

# Total weight capacity of the knapsack.
weight_limit = 14

# Build a concrete Pyomo model (all data known up front, no abstract symbols).
m = pyo.ConcreteModel()

# Index set: the names of the candidate items.
m.things = pyo.Set(initialize=data.keys())

# Decision variable: y[i] = 1 if item i is packed, 0 otherwise (binary => 0/1 only).
m.y = pyo.Var(m.things, within=pyo.Binary)

# Objective: maximize the total value of the items selected.
m.value = pyo.Objective(
    expr=sum(data[i]['value'] * m.y[i] for i in m.things),
    sense=pyo.maximize
)

# Capacity constraint: total weight of chosen items must not exceed the limit.
m.weight = pyo.Constraint(
    expr=sum(data[i]['weight'] * m.y[i] for i in m.things) <= weight_limit
)

In [2]:
# Use GLPK as the MILP solver (must be installed and on PATH).
solver = pyo.SolverFactory('glpk')

# Solve the model; tee=True streams the solver's log to stdout.
results = solver.solve(m, tee=True)

print("\nOptimal things:")

# Print every item the solver picked. The 0.5 threshold guards against
# tiny floating-point noise around an integer 0/1 value.
for i in m.things:
    if pyo.value(m.y[i]) > 0.5:
        print(i)

# Optimal objective value (sum of values of chosen items).
print("\nTotal value =", pyo.value(m.value))

# Sanity check: total weight of the chosen items should be <= weight_limit.
print(
    "Total weight =",
    sum(data[i]['weight'] * pyo.value(m.y[i]) for i in m.things)
)

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write C:\Users\Devin\AppData\Local\Temp\tmpuuoj_pwi.glpk.raw --wglp C:\Users\Devin\AppData\Local\Temp\tmpmgedt74_.glpk.glp
 --cpxlp C:\Users\Devin\AppData\Local\Temp\tmpa3iay3z3.pyomo.lp
Reading problem data from 'C:\Users\Devin\AppData\Local\Temp\tmpa3iay3z3.pyomo.lp'...
C:\Users\Devin\AppData\Local\Temp\tmpa3iay3z3.pyomo.lp:43: warning: lower bound of variable 'x2' redefined
C:\Users\Devin\AppData\Local\Temp\tmpa3iay3z3.pyomo.lp:43: warning: upper bound of variable 'x2' redefined
1 row, 10 columns, 10 non-zeros
10 integer variables, all of which are binary
53 lines were read
Writing problem data to 'C:\Users\Devin\AppData\Local\Temp\tmpmgedt74_.glpk.glp'...
35 lines were written
GLPK Integer Optimizer 5.0
1 row, 10 columns, 10 non-zeros
10 integer variables, all of which are binary
Preprocessing...
1 row, 10 columns, 10 non-zeros
10 integer variables, all of which are binary
Scaling...
 A: min|aij| =  1.000